# Faz 1b — cubic_flux Uzun-Ufuk Karar Deneyi

**Projenin asıl özgün iddiasının karar noktası.**

cubic+additive zaten kaybetti (Ek 4). Doğru test: cubic'in platosunun teorik avantaj taşıdığı yer → **seyrek (P=8) + uzun gap + dpfp** (kanalları seyrek tutar → plato korunur).

## Tasarım
- **4 konfigürasyon:** {exp, cubic_flux_chunked} × {elu, dpfp}
- **3 LR taraması:** {3e-4, 1e-3, 3e-3} (mod-başına en iyi LR seçilir)
- **3 seed:** {0, 1, 2}
- **Eğitim:** ctx=160, 600 adım, P=8 dense recall
- **Değerlendirme:** ctx=640 ve ctx=1280
- **Toplam koşu:** 4 × 3 × 3 = **36 koşu** (2× T4 paralel → ~15 dk)

## Önceden Kayıtlı Başarı Kriteri
> cubic_flux_chunked + dpfp, **256+ kovasında** exp + dpfp'yi 3 seed ortalamasında **>2 standart hata** geçerse → **cubic uzun-ufuk avantajı DOĞRULANDI.**
> Aksi halde hipotez reddedilir / parked kalır.

In [ ]:
# --- KURULUM ---
import subprocess, sys, os
REPO = "/kaggle/working/HFP"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "https://github.com/kayra-hn/HFP.git", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers>=4.40"], check=True)
os.chdir(REPO)
sys.path.insert(0, REPO)
os.environ["HFP_CKPT_DIR"] = "/kaggle/working/ckpts"
os.environ["PYTHONPATH"] = REPO
os.makedirs("/kaggle/working/ckpts", exist_ok=True)

import torch
num_gpus = torch.cuda.device_count()
print(f"GPU sayisi: {num_gpus}")
for i in range(num_gpus):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
print("Kurulum tamam!")

In [ ]:
# --- 36 KOSU: 4 konfig x 3 LR x 3 seed --- 2x T4 PARALEL ---
import subprocess, sys, os, time
from concurrent.futures import ThreadPoolExecutor, as_completed
import torch

retentions = ["exp", "cubic_flux_chunked"]
fmaps = ["elu", "dpfp"]
lrs = [3e-4, 1e-3, 3e-3]
seeds = [0, 1, 2]
BUDGET = 600  # saniye (script icinde step=600, bu zaman limiti)

# Tum kosu konfigurasyonlarini olustur
all_runs = []
for ret in retentions:
    for fmap in fmaps:
        for lr in lrs:
            for seed in seeds:
                tag = f"{ret}_{fmap}_lr{lr:g}_s{seed}"
                all_runs.append((ret, fmap, lr, seed, tag))

print(f"Toplam {len(all_runs)} kosu planlanadi.")
num_gpus = torch.cuda.device_count()
print(f"Kullanilacak GPU sayisi: {num_gpus}")

def run_single(ret, fmap, lr, seed, tag, gpu_id):
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
    cmd = [
        sys.executable, "review_scripts/cubic_longhorizon.py",
        ret, fmap, str(lr), str(seed), str(BUDGET)
    ]
    t0 = time.time()
    result = subprocess.run(cmd, capture_output=True, text=True, env=env)
    elapsed = time.time() - t0
    status = "OK" if result.returncode == 0 else "HATA"
    stdout = result.stdout[-1500:] if len(result.stdout) > 1500 else result.stdout
    stderr = result.stderr[-500:] if result.returncode != 0 else ""
    return tag, gpu_id, elapsed, status, stdout, stderr

total_start = time.time()
completed = 0
workers = max(1, num_gpus)

with ThreadPoolExecutor(max_workers=workers) as executor:
    futures = {}
    for i, (ret, fmap, lr, seed, tag) in enumerate(all_runs):
        gpu_id = i % workers
        future = executor.submit(run_single, ret, fmap, lr, seed, tag, gpu_id)
        futures[future] = tag
    
    for future in as_completed(futures):
        tag, gpu_id, elapsed, status, stdout, stderr = future.result()
        completed += 1
        print(f"\n[{completed}/{len(all_runs)}] {tag} (GPU {gpu_id}) - {status} - {elapsed:.1f}s")
        # Sadece eval sonuclarini goster (son satirlar)
        for line in stdout.strip().split('\n'):
            if 'eval' in line or 'BITTI' in line or 'loss' in line.lower():
                print(f"  {line.strip()}")
        if stderr:
            print(f"  HATA: {stderr}")

total_elapsed = time.time() - total_start
print(f"\n{'='*70}")
print(f"  TUM {len(all_runs)} KOSU TAMAMLANDI - Toplam: {total_elapsed:.0f}s ({total_elapsed/60:.1f} dk)")
print(f"{'='*70}")

In [ ]:
# --- SONUCLARI AYRISTIR VE KARAR KRITERINI UYGULA ---
import ast, re, os
import numpy as np

results_file = "/kaggle/working/ckpts/cubic_lh_results.txt"
assert os.path.exists(results_file), f"{results_file} bulunamadi!"

# Parse: "clh_exp_dpfp_0.001_0 ctx640 {'<48': 71.7, ...}"
data = []  # list of dicts
with open(results_file) as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        # TAG formatı: clh_<ret>_<fmap>_<lr>_<seed> ctx<N> {dict}
        m = re.match(r'clh_(\S+?)_(elu|dpfp)_([\d.e-]+)_(\d+)\s+ctx(\d+)\s+(\{.+\})', line)
        if m:
            ret, fmap, lr, seed, ctx, buckets_str = m.groups()
            buckets = ast.literal_eval(buckets_str)
            data.append({
                'retention': ret, 'fmap': fmap, 'lr': float(lr),
                'seed': int(seed), 'ctx': int(ctx), **buckets
            })

print(f"Toplam {len(data)} eval kaydi okundu.")

# --- DETAYLI TABLO ---
print(f"\n{'='*90}")
print(f"  TUM SONUCLAR (eval ctx = 640 ve 1280)")
print(f"{'='*90}")
print(f"{'Retention':<25} {'FMap':<6} {'LR':<8} {'Seed':<5} {'Ctx':<6} {'<48':>6} {'48-127':>8} {'128-255':>9} {'256+':>6}")
print('-'*90)
for d in sorted(data, key=lambda x: (x['retention'], x['fmap'], x['lr'], x['seed'], x['ctx'])):
    print(f"{d['retention']:<25} {d['fmap']:<6} {d['lr']:<8g} {d['seed']:<5} {d['ctx']:<6} {d.get('<48',0):>6.1f} {d.get('48-127',0):>8.1f} {d.get('128-255',0):>9.1f} {d.get('256+',0):>6.1f}")

# --- MOD BASINA EN IYI LR SECIMI ---
print(f"\n{'='*90}")
print(f"  MOD BASINA EN IYI LR SECIMI (256+ kovasi, ctx=1280, 3-seed ortalamasi)")
print(f"{'='*90}")

configs = {}
for d in data:
    if d['ctx'] != 1280:
        continue
    key = (d['retention'], d['fmap'], d['lr'])
    configs.setdefault(key, []).append(d.get('256+', 0.0))

best_lr = {}
for (ret, fmap, lr), vals in sorted(configs.items()):
    mean_val = np.mean(vals)
    std_val = np.std(vals, ddof=1) if len(vals) > 1 else 0
    se_val = std_val / np.sqrt(len(vals)) if len(vals) > 1 else 0
    n = len(vals)
    print(f"  {ret:<25} {fmap:<6} lr={lr:<8g} n={n} 256+: {mean_val:6.1f} ± {se_val:.1f} (SE)")
    cfg_key = (ret, fmap)
    if cfg_key not in best_lr or mean_val > best_lr[cfg_key][1]:
        best_lr[cfg_key] = (lr, mean_val, se_val, vals)

print(f"\n--- Secilen en iyi LR'ler ---")
for (ret, fmap), (lr, mean_val, se, vals) in sorted(best_lr.items()):
    print(f"  {ret} + {fmap}: LR={lr:g} -> 256+ = {mean_val:.1f} ± {se:.1f}")

# --- KARAR KRITERI: cubic+dpfp vs exp+dpfp ---
print(f"\n{'='*90}")
print(f"  ONCEDEN KAYITLI KARAR KRITERI")
print(f"  'cubic+dpfp 256+ > exp+dpfp 256+ (>2 SE farkla) -> DOGRULANDI'")
print(f"{'='*90}")

cubic_dpfp = best_lr.get(('cubic_flux_chunked', 'dpfp'))
exp_dpfp = best_lr.get(('exp', 'dpfp'))

if cubic_dpfp and exp_dpfp:
    c_lr, c_mean, c_se, c_vals = cubic_dpfp
    e_lr, e_mean, e_se, e_vals = exp_dpfp
    
    diff = c_mean - e_mean
    # Birlesik SE (bagimsiz orneklem)
    pooled_se = np.sqrt(c_se**2 + e_se**2) if (c_se > 0 or e_se > 0) else 0.01
    ratio = diff / pooled_se if pooled_se > 0 else 0
    
    print(f"\n  cubic+dpfp (LR={c_lr:g}): {c_mean:.1f} ± {c_se:.1f}  seeds: {c_vals}")
    print(f"  exp+dpfp   (LR={e_lr:g}): {e_mean:.1f} ± {e_se:.1f}  seeds: {e_vals}")
    print(f"  Fark: {diff:+.1f}  Birlesik SE: {pooled_se:.1f}  Fark/SE: {ratio:.2f}")
    print()
    
    if diff > 0 and ratio > 2.0:
        print("  ✅ KARAR: cubic_flux uzun-ufuk avantaji DOGRULANDI")
        print(f"     (fark = {diff:.1f}, {ratio:.1f}× SE > 2 SE esigi)")
    elif diff > 0:
        print("  ⚠️  KARAR: cubic onunde ama fark <2 SE — BELIRSIZ")
        print(f"     (fark = {diff:.1f}, {ratio:.1f}× SE < 2 SE esigi)")
    else:
        print("  ❌ KARAR: cubic_flux uzun-ufuk avantaji REDDEDILDI")
        print(f"     (exp onunde, fark = {abs(diff):.1f})")
else:
    print("  HATA: cubic+dpfp veya exp+dpfp sonuclari eksik!")

# --- EK: elu sonuclari (ablasyon) ---
print(f"\n{'='*90}")
print(f"  EK: dpfp vs elu etkilesimi (ctx=1280, 256+ kovasi)")
print(f"{'='*90}")
for (ret, fmap), (lr, mean_val, se, vals) in sorted(best_lr.items()):
    print(f"  {ret:<25} + {fmap:<6} (LR={lr:g}): {mean_val:.1f} ± {se:.1f}")

In [ ]:
# --- TUM KOVALARI GORSEL TABLO OLARAK GOSTER ---
import numpy as np

print(f"\n{'='*90}")
print(f"  NIHAI ADIL KARSILASTIRMA — EN IYI LR, 3-SEED ORTALAMA (ctx=1280)")
print(f"{'='*90}")
print(f"{'Konfigurasyon':<35} {'LR':<8} {'<48':>8} {'48-127':>10} {'128-255':>10} {'256+':>8}")
print('-'*90)

for (ret, fmap), (lr, _, _, _) in sorted(best_lr.items()):
    bucket_means = {}
    for d in data:
        if d['ctx'] != 1280 or d['retention'] != ret or d['fmap'] != fmap or d['lr'] != lr:
            continue
        for b in ['<48', '48-127', '128-255', '256+']:
            bucket_means.setdefault(b, []).append(d.get(b, 0))
    
    label = f"{ret} + {fmap}"
    vals_str = ""
    for b in ['<48', '48-127', '128-255', '256+']:
        vs = bucket_means.get(b, [0])
        m = np.mean(vs)
        s = np.std(vs, ddof=1) / np.sqrt(len(vs)) if len(vs) > 1 else 0
        vals_str += f"{m:7.1f}±{s:.1f} "
    print(f"  {label:<33} {lr:<8g} {vals_str}")

print(f"\n(sans seviyesi: %3.3)")
print(f"\nBu sonuclari DENEY_SONUCLARI.md'ye ekleyin.")